In [ ]:
# Code with Precheck/Settings Report/Better Soma Detection
# + optional soma max area filter
# + optional soma circularity filter

import os
import math
import re
import numpy as np
import pandas as pd
from tqdm import tqdm

from skimage import io, img_as_float, exposure
from skimage.filters import median, threshold_otsu, gaussian, frangi
from skimage.morphology import (
    skeletonize,
    remove_small_objects,
    remove_small_holes,
    closing,
    disk,
    dilation,
)
from skimage.measure import label, regionprops, perimeter
import scipy.ndimage as ndi
from scipy.spatial import cKDTree
from skimage.graph import route_through_array
import networkx as nx
import matplotlib.pyplot as plt


# =========================
# MULTI-EXPERIMENT SETTINGS
# =========================
ROOT_DIR = r"D:/All Neuroprotections/New Soma Detection/Input/"
OUTPUT_ROOT = r"D:/All Neuroprotections/New Soma Detection/Output with TH480 Repeat/"

USE_PER_FOLDER_PLATEMAP = True
GLOBAL_PLATEMAP_PATH = r"D:/Python/Neuroprotection Analysis/platemap.csv"
PER_FOLDER_PLATEMAP_CANDIDATES = ["platemap.csv", "platemap.xlsx", "PlateMap.csv", "PlateMap.xlsx"]

MIN_IMAGES_TO_PROCESS = 1
EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}


# =========================
# Output / Plot Settings
# =========================
MAKE_PLOTS = True
PLOT_REPLICATE_BARS = True
PLOT_CONDITION_MEAN = True
DPI_PLOTS = 200


# =========================
# PRE-CHECK SETTINGS
# =========================
ENABLE_PRECHECK = False
N_SAMPLE_FOR_PRECHECK = 60
STOP_AFTER_PRECHECK = True


# =========================
# Composite Score Weights
# =========================
REPAIR_W_NEURITE = 0.6
REPAIR_W_CELLS = 0.4


# =========================
# Pre-processing
# =========================
PERC_CLIP = (0.00005, 99.99995)
GAMMA = 0.3
MEDIAN_RADIUS = 1
GAUSSIAN_SIGMA = 0.01

# Vesselness
USE_VESSELNESS = False
VESSEL_SIGMAS = (0.075, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0, 1.4, 1.6, 1.8, 2.0, 3.0)
VESSEL_WEIGHT = 0.75

# Thresholding
THRESHOLD_BIAS = 0.15
GLOBAL_MIN_T = 0.005
GLOBAL_MAX_T = None

# Binary cleanup
APPLY_VESSEL_MASK = False
VESSEL_MIN_RESP = 0.06
MIN_OBJECT_SIZE = 4
MIN_HOLE_AREA = 12
MORPH_CLOSING = 2

# Soma/thick detection
DETECT_SOMA = True
SOMA_PERC = 97.0
SOMA_MIN_AREA = 60
SOMA_DILATE = 3
SOMA_THICK_RADIUS = 12

# NEW: Soma artifact filters
USE_SOMA_MAX_AREA_FILTER = False
SOMA_MAX_AREA = 4000

USE_SOMA_CIRCULARITY_FILTER = True
SOMA_MIN_CIRCULARITY = None   # e.g. 0.15
SOMA_MAX_CIRCULARITY = 0.70   # e.g. 0.90

# Bridging
BRIDGE_SMALL_GAPS = True
BRIDGE_MAX_GAP = 25
BRIDGE_MIN_MEAN = 0.005
BRIDGE_USE_VESSEL = True

# Pruning
MAX_RADIUS_FOR_SKELETON = 12.0
MIN_SPUR_LEN_PX = 8.0
MIN_SEG_LEN_PX = 16.0
POST_PRUNE_BRIDGE_RADIUS = 2

# Overlay
LINE_THICKNESS = 3
ALPHA = 0.85
COLOR_NEURITE = (255, 64, 64)
COLOR_SOMA = (64, 200, 255)

# Filtering (QC)
USE_FILTER = False
FILTER_MODE = "density"
ENDPOINTS_MIN = 150
ENDPOINTS_MAX = 1500
ENDPOINTS_PER_MPX_MIN = 120
ENDPOINTS_PER_MPX_MAX = None


# =========================
# Metrics to summarize/plot
# =========================
METRICS_TO_SUMMARY = [
    "Neurite_TotalLength(px)",
    "ForegroundArea(px)",
    "SomaArea(px)",
    "NeuriteArea(px)",
    "CellCount",
    "NeuriteCoverage",
    "BranchesPerCell",
    "TotalRepairScore",
]


# ============================================================
# Robust filename parsing
# ============================================================
FULL_RE = re.compile(
    r"""
    ^(?P<prefix>.+?)_
    S(?P<site>\d{4})
    \(
        P(?P<pos>\d+)-(?P<well>[A-H]\d{1,2})
    \)
    _C(?P<channel>\d{2})
    (?:\((?P<channel_name>[^)]+)\))?
    _M(?P<m>\d{4})
    _(?P<tag>[^.]+?)
    (?:\.[^.]+)?$
    """,
    re.IGNORECASE | re.VERBOSE,
)

FALLBACK_RE = re.compile(r"P(?P<pos>\d+)-(?P<well>[A-H]\d{1,2})", re.IGNORECASE)


def parse_filename_meta(path: str) -> dict:
    base = os.path.basename(path)
    name = os.path.splitext(base)[0]

    m = FULL_RE.match(name)
    if m:
        d = m.groupdict()
        return {
            "BaseName": name,
            "Prefix": d["prefix"],
            "Well": d["well"].upper(),
            "Position": int(d["pos"]),
            "Site": int(d["site"]),
            "Channel": int(d["channel"]),
            "ChannelName": d.get("channel_name", None),
            "M": int(d["m"]),
            "Tag": d["tag"],
            "ParseError": "",
        }

    m2 = FALLBACK_RE.search(name)
    if m2:
        return {
            "BaseName": name,
            "Prefix": None,
            "Well": m2.group("well").upper(),
            "Position": int(m2.group("pos")),
            "Site": None,
            "Channel": None,
            "ChannelName": None,
            "M": None,
            "Tag": None,
            "ParseError": "",
        }

    return {
        "BaseName": name,
        "Prefix": None,
        "Well": None,
        "Position": None,
        "Site": None,
        "Channel": None,
        "ChannelName": None,
        "M": None,
        "Tag": None,
        "ParseError": "Could not extract (P#-Well)",
    }


# ============================================================
# Image processing helpers
# ============================================================
def read_grayscale(path):
    img = io.imread(path)
    if img.ndim == 3:
        if img.shape[-1] == 3:
            img = 0.2126 * img[..., 0] + 0.7152 * img[..., 1] + 0.0722 * img[..., 2]
        else:
            img = img.mean(axis=-1)
    return img_as_float(img)


def pre_contrast(img):
    lo, hi = np.percentile(img, PERC_CLIP)
    if hi - lo > 1e-6:
        img = exposure.rescale_intensity(img, in_range=(lo, hi), out_range=(0, 1))
    if abs(GAMMA - 1.0) > 1e-6:
        img = exposure.adjust_gamma(img, gamma=GAMMA)
    return img


def apply_vesselness(img):
    if not USE_VESSELNESS:
        return img, None
    v = frangi(img.astype(np.float32), sigmas=VESSEL_SIGMAS, beta=0.5, gamma=15, black_ridges=False)
    v = np.nan_to_num(v, nan=0.0)
    v = (v - v.min()) / (v.max() - v.min() + 1e-8)
    blended = np.clip((1.0 - VESSEL_WEIGHT) * img + VESSEL_WEIGHT * v, 0, 1)
    return blended, v


def compute_circularity(mask: np.ndarray) -> float:
    area = float(mask.sum())
    if area <= 0:
        return 0.0
    perim = float(perimeter(mask, neighborhood=8))
    if perim <= 0:
        return 0.0
    return float((4.0 * math.pi * area) / (perim ** 2))


def filter_soma_components(soma_mask: np.ndarray) -> np.ndarray:
    """
    Optional filtering of connected soma components by:
      - maximum area
      - circularity range
    """
    if not USE_SOMA_MAX_AREA_FILTER and not USE_SOMA_CIRCULARITY_FILTER:
        return soma_mask

    lab = label(soma_mask)
    out = np.zeros_like(soma_mask, dtype=bool)

    for reg in regionprops(lab):
        comp = (lab == reg.label)
        area = int(reg.area)

        keep = True

        if USE_SOMA_MAX_AREA_FILTER and SOMA_MAX_AREA is not None:
            if area > SOMA_MAX_AREA:
                keep = False

        if keep and USE_SOMA_CIRCULARITY_FILTER:
            circ = compute_circularity(comp)

            if SOMA_MIN_CIRCULARITY is not None and circ < SOMA_MIN_CIRCULARITY:
                keep = False
            if SOMA_MAX_CIRCULARITY is not None and circ > SOMA_MAX_CIRCULARITY:
                keep = False

        if keep:
            out |= comp

    return out


def get_soma_intensity_mask(img):
    pre0 = pre_contrast(img)
    thr_soma = np.percentile(pre0, SOMA_PERC)
    soma_int = pre0 >= thr_soma
    if SOMA_MIN_AREA > 0:
        soma_int = remove_small_objects(soma_int, SOMA_MIN_AREA)
    if SOMA_DILATE > 0:
        soma_int = dilation(soma_int, disk(int(SOMA_DILATE)))
    return soma_int


def binarize(img):
    """
    Returns:
      bw_all: bool mask including soma + neurites
      t: threshold
      vesselness: vesselness image (or None)
      soma_mask: bool mask for soma/thick areas
    """
    pre0 = pre_contrast(img)
    work = pre0.copy()

    if MEDIAN_RADIUS > 0:
        work = median(work, footprint=disk(int(MEDIAN_RADIUS)))

    work, vesselness = apply_vesselness(work)

    if GAUSSIAN_SIGMA > 0:
        work = gaussian(work, sigma=GAUSSIAN_SIGMA, preserve_range=True)

    t = threshold_otsu(work)
    t = max(0.0, t - THRESHOLD_BIAS)
    if GLOBAL_MIN_T is not None:
        t = max(t, GLOBAL_MIN_T)
    if GLOBAL_MAX_T is not None:
        t = min(t, GLOBAL_MAX_T)

    bw = work >= t

    if APPLY_VESSEL_MASK and vesselness is not None:
        bw &= (vesselness >= VESSEL_MIN_RESP)

    if MIN_OBJECT_SIZE > 0:
        bw = remove_small_objects(bw, min_size=MIN_OBJECT_SIZE)
    if MIN_HOLE_AREA > 0:
        bw = remove_small_holes(bw, area_threshold=MIN_HOLE_AREA)
    if MORPH_CLOSING > 0:
        bw = closing(bw, disk(int(MORPH_CLOSING)))

    soma_mask = np.zeros_like(bw, dtype=bool)
    if DETECT_SOMA:
        soma_int = get_soma_intensity_mask(img)
        dist = ndi.distance_transform_edt(bw)
        soma_thick = dist >= float(SOMA_THICK_RADIUS)
        soma_mask = soma_int | soma_thick
        soma_mask = filter_soma_components(soma_mask)

    return bw, t, vesselness, soma_mask


def bridge_gaps(cost_src, skel):
    if not BRIDGE_SMALL_GAPS:
        return skel
    conv = ndi.convolve(skel.astype(np.uint8), np.ones((3, 3)), mode='constant', cval=0)
    ep_mask = skel & (conv == 1)
    pts = np.argwhere(ep_mask)
    if len(pts) < 2:
        return skel

    src = (cost_src - cost_src.min()) / (cost_src.max() - cost_src.min() + 1e-8)
    cost = 1.0 - src
    pairs = cKDTree(pts).query_pairs(r=float(BRIDGE_MAX_GAP))
    out = skel.copy()

    for i, j in pairs:
        p1, p2 = tuple(pts[i]), tuple(pts[j])
        try:
            _, path = route_through_array(cost, p1, p2, fully_connected=True, geometric=True)
            path = np.asarray(path, int)
            if src[path[:, 0], path[:, 1]].mean() >= float(BRIDGE_MIN_MEAN):
                out[path[:, 0], path[:, 1]] = True
        except Exception:
            pass
    return out


def build_graph(skel):
    coords = np.argwhere(skel)
    G = nx.Graph()
    for rc in map(tuple, coords):
        G.add_node(rc)
    for r, c in map(tuple, coords):
        for dr, dc in [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]:
            nb = (r + dr, c + dc)
            if nb in G:
                G.add_edge((r, c), nb, weight=math.hypot(dr, dc))
    return G


def prune_skeleton(bw_for_radius, skel):
    if not skel.any():
        return skel

    dist = ndi.distance_transform_edt(bw_for_radius)
    skel = skel & (dist <= float(MAX_RADIUS_FOR_SKELETON) + 1e-6)

    G = build_graph(skel)
    deg = dict(G.degree())
    junctions = [n for n, d in deg.items() if d != 2]
    visited = set()
    keep = np.zeros_like(skel, bool)

    def walk(u, v):
        prev, curr = u, v
        L = G[u][v]['weight']
        visited.add(frozenset({u, v}))
        nodes = [u, v]
        while deg.get(curr, 0) == 2:
            nxt = next(n for n in G.neighbors(curr) if n != prev)
            e = frozenset({curr, nxt})
            if e in visited:
                break
            visited.add(e)
            L += G[curr][nxt]['weight']
            nodes.append(nxt)
            prev, curr = curr, nxt
        return nodes, L, (u, curr)

    segs = []
    for j in junctions:
        for nb in G.neighbors(j):
            e = frozenset({j, nb})
            if e not in visited:
                segs.append(walk(j, nb))
    for u, v in G.edges():
        e = frozenset({u, v})
        if e not in visited:
            segs.append(walk(u, v))

    for nodes, L, (u, v) in segs:
        du, dv = deg.get(u, 0), deg.get(v, 0)
        is_spur = (du == 1 and dv >= 3) or (dv == 1 and du >= 3)
        if (is_spur and L < float(MIN_SPUR_LEN_PX)) or (L < float(MIN_SEG_LEN_PX)):
            continue
        for r, c in nodes:
            keep[r, c] = True

    if POST_PRUNE_BRIDGE_RADIUS > 0:
        keep = skeletonize(dilation(keep, disk(int(POST_PRUNE_BRIDGE_RADIUS))))
    return keep


def compute_metrics(skel):
    if not skel.any():
        return 0.0, 0, 0.0, 0, 0, 0

    G = build_graph(skel)
    deg = dict(G.degree())
    endpoints = sum(1 for d in deg.values() if d == 1)
    branches = sum(1 for d in deg.values() if d >= 3)

    visited = set()
    total = 0.0
    segs = 0

    def walk(u, v):
        nonlocal total, segs
        prev, curr = u, v
        L = G[u][v]['weight']
        visited.add(frozenset({u, v}))
        while deg.get(curr, 0) == 2:
            nxt = next(n for n in G.neighbors(curr) if n != prev)
            e = frozenset({curr, nxt})
            if e in visited:
                break
            visited.add(e)
            L += G[curr][nxt]['weight']
            prev, curr = curr, nxt
        total += L
        segs += 1

    for n, d in deg.items():
        if d != 2:
            for nb in G.neighbors(n):
                e = frozenset({n, nb})
                if e not in visited:
                    walk(n, nb)

    for u, v in G.edges():
        e = frozenset({u, v})
        if e not in visited:
            walk(u, v)

    avg = total / segs if segs > 0 else 0.0
    return total, segs, avg, branches, endpoints, int(skel.sum())


def make_overlay(img, skel, soma_mask=None):
    base = (np.clip(img, 0, 1) * 255).astype(np.uint8)
    rgb = np.dstack([base] * 3).astype(np.float32) / 255.0

    if LINE_THICKNESS > 1:
        sk_draw = dilation(skel, disk(int(max(1, LINE_THICKNESS // 2))))
    else:
        sk_draw = skel

    color_neur = np.array(COLOR_NEURITE) / 255.0
    m = sk_draw[..., None].astype(np.float32)
    rgb = rgb * (1.0 - m * ALPHA) + color_neur * (m * ALPHA)

    if soma_mask is not None:
        soma_draw = dilation(soma_mask, disk(2))
        color_soma = np.array(COLOR_SOMA) / 255.0
        m2 = soma_draw[..., None].astype(np.float32)
        rgb = rgb * (1.0 - m2 * 0.50) + color_soma * (m2 * 0.50)

    return (np.clip(rgb, 0, 1) * 255).astype(np.uint8)


def passes_bandpass(row):
    if not USE_FILTER:
        return True, ""
    if "Error" in row and pd.notna(row["Error"]):
        return False, "ProcessingError"

    if FILTER_MODE.lower() == "density":
        val = row.get("Neurite_Endpoints_per_MPx", None)
        lo, hi = ENDPOINTS_PER_MPX_MIN, ENDPOINTS_PER_MPX_MAX
        metric_name = "Neurite_Endpoints_per_MPx"
    else:
        val = row.get("Neurite_Endpoints", None)
        lo, hi = ENDPOINTS_MIN, ENDPOINTS_MAX
        metric_name = "Neurite_Endpoints"

    if val is None or (isinstance(val, float) and np.isnan(val)):
        return False, f"Missing{metric_name}"
    if lo is not None and val < lo:
        return False, f"{metric_name}<{lo}"
    if hi is not None and val > hi:
        return False, f"{metric_name}>{hi}"
    return True, ""


# ============================================================
# Composite score helpers
# ============================================================
def zscore_series(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    mu = s.mean()
    sd = s.std(ddof=1)
    if sd == 0 or np.isnan(sd):
        return (s - mu) * 0.0
    return (s - mu) / sd


def add_total_repair_score(df: pd.DataFrame, w_neurite: float, w_cells: float) -> pd.DataFrame:
    out = df.copy()
    if "Neurite_TotalLength(px)" not in out.columns or "ForegroundArea(px)" not in out.columns:
        return out
    zL = zscore_series(out["Neurite_TotalLength(px)"])
    zA = zscore_series(out["ForegroundArea(px)"])
    out["TotalRepairScore"] = w_neurite * zL + w_cells * zA
    return out


# ============================================================
# Per-image processing
# ============================================================
def process_one(path):
    img = read_grayscale(path)
    bw_all, t, vesselness, soma_mask = binarize(img)

    bw_neurite = bw_all & (~soma_mask)

    skel = skeletonize(bw_neurite)
    cost_src = vesselness if (BRIDGE_USE_VESSEL and vesselness is not None) else pre_contrast(img)
    skel = bridge_gaps(cost_src, skel)
    skel = prune_skeleton(bw_neurite, skel)

    neur_total_len_px, neur_n_seg, neur_avg_len_px, neur_n_branch, neur_n_end, neur_n_pix = compute_metrics(skel)

    H, W = img.shape[:2]
    area_px = int(H * W)
    neur_endpoints_per_mpx = (neur_n_end / max(area_px, 1)) * 1_000_000.0

    foreground_area = int(bw_all.sum())
    soma_area = int((bw_all & soma_mask).sum())
    neurite_area = int(bw_neurite.sum())

    _, n_cells = ndi.label(soma_mask)
    cell_count = int(n_cells)

    foreground_coverage = foreground_area / max(area_px, 1)
    neurite_coverage = neurite_area / max(foreground_area, 1)
    branches_per_cell = neur_n_branch / max(cell_count, 1)

    metrics = {
        "Image": os.path.basename(path),
        "Height": H,
        "Width": W,
        "ImageArea(px)": area_px,
        "Threshold(Otsu-bias)": t,
        "Neurite_SkeletonPixels": neur_n_pix,
        "Neurite_BranchPoints": neur_n_branch,
        "Neurite_Endpoints": neur_n_end,
        "Neurite_Endpoints_per_MPx": neur_endpoints_per_mpx,
        "Neurite_TotalSegments": neur_n_seg,
        "Neurite_TotalLength(px)": neur_total_len_px,
        "Neurite_AvgSegmentLength(px)": neur_avg_len_px,
        "ForegroundArea(px)": foreground_area,
        "SomaArea(px)": soma_area,
        "NeuriteArea(px)": neurite_area,
        "CellCount": cell_count,
        "ForegroundCoverage": foreground_coverage,
        "NeuriteCoverage": neurite_coverage,
        "BranchesPerCell": branches_per_cell,
        "TotalRepairScore": np.nan,
    }

    metrics.update(parse_filename_meta(path))
    overlay = make_overlay(img, skel, soma_mask)
    return overlay, metrics


# ============================================================
# PRE-CHECK helpers
# ============================================================
def quick_density_score(path):
    img = read_grayscale(path)
    pre0 = pre_contrast(img)
    work = pre0.copy()

    if MEDIAN_RADIUS > 0:
        work = median(work, footprint=disk(int(MEDIAN_RADIUS)))

    work2, vessel = apply_vesselness(work)

    if GAUSSIAN_SIGMA > 0:
        work2 = gaussian(work2, sigma=GAUSSIAN_SIGMA, preserve_range=True)

    t = threshold_otsu(work2)
    t = max(0.0, t - THRESHOLD_BIAS)
    if GLOBAL_MIN_T is not None:
        t = max(t, GLOBAL_MIN_T)
    if GLOBAL_MAX_T is not None:
        t = min(t, GLOBAL_MAX_T)

    bw = work2 >= t
    if APPLY_VESSEL_MASK and vessel is not None:
        bw &= (vessel >= VESSEL_MIN_RESP)

    return float(bw.mean())


def pick_representative_images(image_paths, n_sample=60):
    if len(image_paths) == 0:
        return []

    if len(image_paths) <= n_sample:
        sample = image_paths
    else:
        sample = list(np.random.choice(image_paths, n_sample, replace=False))

    scores = []
    for p in sample:
        try:
            scores.append((quick_density_score(p), p))
        except Exception:
            continue

    if len(scores) < 3:
        scores.sort(key=lambda x: x[0])
        return [p for _, p in scores]

    scores.sort(key=lambda x: x[0])
    return [scores[0][1], scores[len(scores) // 2][1], scores[-1][1]]


def show_precheck(representative_paths, title="Pre-check"):
    if not representative_paths:
        print("Pre-check: no images available to preview.")
        return

    n = len(representative_paths)
    fig, axes = plt.subplots(n, 6, figsize=(21, 4 * n))
    if n == 1:
        axes = np.array([axes])

    for i, p in enumerate(representative_paths):
        try:
            img = read_grayscale(p)
            bw_all, t, vesselness, soma_mask = binarize(img)
            soma_int = get_soma_intensity_mask(img)
            bw_neurite = bw_all & (~soma_mask)

            skel = skeletonize(bw_neurite)
            cost_src = vesselness if (BRIDGE_USE_VESSEL and vesselness is not None) else pre_contrast(img)
            skel = bridge_gaps(cost_src, skel)
            skel = prune_skeleton(bw_neurite, skel)
            ov = make_overlay(img, skel, soma_mask)

            axes[i, 0].imshow(img, cmap="gray")
            axes[i, 0].set_title(f"Original\n{os.path.basename(p)}")
            axes[i, 0].axis("off")

            if vesselness is not None:
                axes[i, 1].imshow(vesselness, cmap="gray", vmin=0, vmax=1)
                axes[i, 1].set_title("Vesselness")
            else:
                axes[i, 1].imshow(np.zeros_like(img), cmap="gray", vmin=0, vmax=1)
                axes[i, 1].set_title("Vesselness OFF")
            axes[i, 1].axis("off")

            axes[i, 2].imshow(bw_all, cmap="gray", vmin=0, vmax=1)
            axes[i, 2].set_title(f"Binary (bw_all)\nt={t:.4f}")
            axes[i, 2].axis("off")

            axes[i, 3].imshow(soma_int, cmap="gray", vmin=0, vmax=1)
            axes[i, 3].set_title("Soma intensity mask")
            axes[i, 3].axis("off")

            axes[i, 4].imshow(soma_mask, cmap="gray", vmin=0, vmax=1)
            axes[i, 4].set_title("Final soma mask")
            axes[i, 4].axis("off")

            axes[i, 5].imshow(ov)
            axes[i, 5].set_title("Overlay (neurite+soma)")
            axes[i, 5].axis("off")

        except Exception as e:
            for j in range(6):
                axes[i, j].text(0.1, 0.5, f"Failed preview:\n{e}", transform=axes[i, j].transAxes)
                axes[i, j].axis("off")

    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()


# ============================================================
# Platemap loading
# ============================================================
def load_platemap(path: str) -> pd.DataFrame:
    if not path or not os.path.exists(path):
        raise FileNotFoundError(f"Platemap not found: {path}")

    ext = os.path.splitext(path)[1].lower()
    if ext in [".xlsx", ".xls"]:
        pm = pd.read_excel(path)
    else:
        try:
            pm = pd.read_csv(path)
        except Exception:
            pm = pd.read_csv(path, sep="\t")

    required = {"Well", "Condition", "Dose", "Replicate"}
    missing = required - set(pm.columns)
    if missing:
        raise ValueError(f"Platemap is missing columns: {sorted(missing)}")

    pm = pm.copy()
    pm["Well"] = pm["Well"].astype(str).str.upper().str.strip()
    pm["Condition"] = pm["Condition"].astype(str).str.strip()
    pm["Dose"] = pd.to_numeric(pm["Dose"], errors="coerce")
    pm["Replicate"] = pd.to_numeric(pm["Replicate"], errors="coerce")
    return pm


# ============================================================
# Batch runner helpers
# ============================================================
def find_experiment_folders(root_dir: str):
    folders = []
    for name in sorted(os.listdir(root_dir)):
        p = os.path.join(root_dir, name)
        if not os.path.isdir(p):
            continue
        imgs = [f for f in os.listdir(p) if os.path.splitext(f)[1].lower() in EXTS]
        if len(imgs) >= MIN_IMAGES_TO_PROCESS:
            folders.append(p)
    return folders


def pick_platemap_for_folder(folder: str) -> str:
    if USE_PER_FOLDER_PLATEMAP:
        for cand in PER_FOLDER_PLATEMAP_CANDIDATES:
            pp = os.path.join(folder, cand)
            if os.path.exists(pp):
                return pp
    return GLOBAL_PLATEMAP_PATH


# ============================================================
# Plotting helpers
# ============================================================
def plot_replicate_bars(per_well_stats: pd.DataFrame, metric: str, outpath: str):
    mean_col = f"{metric}_mean"
    sd_col = f"{metric}_std"
    if mean_col not in per_well_stats.columns:
        return

    rs = per_well_stats.copy()
    rs["Replicate"] = pd.to_numeric(rs["Replicate"], errors="coerce")
    rs = rs.sort_values(["Condition", "Dose", "Replicate"])
    rs["GroupLabel"] = rs["Condition"].astype(str) + " (Dose " + rs["Dose"].astype(str) + ")"

    groups = rs["GroupLabel"].unique().tolist()
    reps = sorted([r for r in rs["Replicate"].dropna().unique().tolist()])

    x = np.arange(len(groups))
    width = 0.25 if len(reps) <= 3 else 0.8 / max(1, len(reps))

    plt.figure(figsize=(max(10, len(groups) * 0.6), 4))

    for k, r in enumerate(reps):
        sub = rs[rs["Replicate"] == r].set_index("GroupLabel").reindex(groups).reset_index()
        y = sub[mean_col].to_numpy()
        if sd_col in sub.columns and sub[sd_col].notna().any():
            e = sub[sd_col].to_numpy()
            plt.bar(x + (k - (len(reps) - 1) / 2) * width, y, width=width, yerr=e, capsize=4, label=f"Rep {int(r)}")
        else:
            plt.bar(x + (k - (len(reps) - 1) / 2) * width, y, width=width, label=f"Rep {int(r)}")

    plt.xticks(x, groups, rotation=45, ha="right")
    plt.ylabel(metric)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=DPI_PLOTS)
    plt.close()


def plot_condition_mean_sem(cond_summary: pd.DataFrame, metric: str, outpath: str):
    mean_col = f"{metric}_mean"
    sem_col = f"{metric}_sem"
    if "GroupLabel" not in cond_summary.columns or mean_col not in cond_summary.columns:
        return

    x = cond_summary["GroupLabel"].astype(str).tolist()
    y = cond_summary[mean_col].to_numpy()

    plt.figure(figsize=(max(10, len(x) * 0.6), 4))
    if sem_col in cond_summary.columns and cond_summary[sem_col].notna().any():
        e = cond_summary[sem_col].to_numpy()
        plt.bar(x, y, yerr=e, capsize=4)
    else:
        plt.bar(x, y)

    plt.xticks(rotation=45, ha="right")
    plt.ylabel(metric)
    plt.tight_layout()
    plt.savefig(outpath, dpi=DPI_PLOTS)
    plt.close()


# ============================================================
# Single experiment runner
# ============================================================
def run_one_experiment(input_dir: str, platemap_path: str, output_base: str):
    exp_name = os.path.basename(os.path.normpath(input_dir))

    out_xlsx = output_base + ".xlsx"
    overlay_dir = output_base + "_overlays"
    plot_dir = output_base + "_plots"

    os.makedirs(overlay_dir, exist_ok=True)
    if MAKE_PLOTS:
        os.makedirs(plot_dir, exist_ok=True)

    image_paths = [
        os.path.join(input_dir, f)
        for f in os.listdir(input_dir)
        if os.path.splitext(f)[1].lower() in EXTS
    ]
    image_paths.sort()

    print(f"\n=== {exp_name} ===")
    print(f"Input: {input_dir}")
    print(f"Images: {len(image_paths)}")
    print(f"Platemap: {platemap_path}")
    print(f"Output: {out_xlsx}")

    if ENABLE_PRECHECK and len(image_paths) > 0:
        reps = pick_representative_images(image_paths, n_sample=N_SAMPLE_FOR_PRECHECK)
        print("Pre-check representatives (low/median/high mask density):")
        for rp in reps:
            print("  ", os.path.basename(rp))
        show_precheck(reps, title=f"{exp_name} PRE-CHECK: tweak THRESHOLD_BIAS / SOMA filters then rerun")

        if STOP_AFTER_PRECHECK:
            raise SystemExit(
                "Pre-check shown. Adjust parameters at top of script, then re-run. "
                "Set STOP_AFTER_PRECHECK=False to proceed with full batch."
            )

    rows = []
    for p in tqdm(image_paths, desc=f"Processing {exp_name}"):
        try:
            overlay, metrics = process_one(p)
            rows.append(metrics)

            base = os.path.splitext(os.path.basename(p))[0]
            out_png = os.path.join(overlay_dir, f"{base}_overlay.png")
            io.imsave(out_png, overlay)
        except Exception as e:
            rows.append({"Image": os.path.basename(p), "Error": repr(e)})
            print(f"Error on {p}: {e}")

    df = pd.DataFrame(rows)
    df = add_total_repair_score(df, REPAIR_W_NEURITE, REPAIR_W_CELLS)

    pass_list, reason_list = [], []
    for _, r in df.iterrows():
        p, why = passes_bandpass(r.to_dict())
        pass_list.append(p)
        reason_list.append(why)
    df["PassFilter"] = pass_list
    df["FilterReason"] = reason_list

    pm = load_platemap(platemap_path)
    df["Well"] = df["Well"].astype(str).str.upper()
    df = df.merge(pm, on=["Well"], how="left", suffixes=("", "_PM"))

    unmapped = df[(df["ParseError"] == "") & (df["Condition"].isna())][
        ["Image", "Well", "BaseName", "Prefix", "Position"]
    ].drop_duplicates()

    parse_fail = df[df["ParseError"] != ""][["Image", "BaseName", "ParseError"]].drop_duplicates()

    mapped = df["Condition"].notna().sum()
    parsed = (df["ParseError"] == "").sum()
    print(f"[{exp_name}] Parsed OK: {parsed}/{len(df)} | Mapped to platemap (by Well): {mapped}/{len(df)}")

    if mapped == 0:
        raise RuntimeError(
            f"[{exp_name}] 0 rows mapped to platemap. Check that your platemap Well column matches filenames (e.g., B2)."
        )

    df_included = df[df["PassFilter"] == True].copy()
    df_excluded = df[df["PassFilter"] == False].copy()

    df_used = df_included.copy()
    df_used = df_used[(df_used["ParseError"] == "") & (df_used["Condition"].notna())]

    metrics_ok = [m for m in METRICS_TO_SUMMARY if m in df_used.columns]

    per_well_avg = None
    per_well_stats = None
    condition_summary = None

    if len(df_used) > 0 and len(metrics_ok) > 0:
        per_well_avg = (
            df_used.groupby(["Well", "Condition", "Dose", "Replicate"], as_index=False)[metrics_ok].mean()
        )

        per_well_stats = (
            df_used.groupby(["Well", "Condition", "Dose", "Replicate"])[metrics_ok].agg(["count", "mean", "std"])
        )
        per_well_stats.columns = ["_".join(c) for c in per_well_stats.columns.to_flat_index()]
        per_well_stats = per_well_stats.reset_index()

        condition_summary = (
            per_well_avg.groupby(["Condition", "Dose"])[metrics_ok].agg(["count", "mean", "std"])
        )
        condition_summary.columns = ["_".join(c) for c in condition_summary.columns.to_flat_index()]
        condition_summary = condition_summary.reset_index()

        for m in metrics_ok:
            condition_summary[f"{m}_sem"] = condition_summary[f"{m}_std"] / np.sqrt(
                condition_summary[f"{m}_count"].clip(lower=1)
            )
        condition_summary["GroupLabel"] = (
            condition_summary["Condition"].astype(str) + " (Dose " + condition_summary["Dose"].astype(str) + ")"
        )

        if MAKE_PLOTS:
            if PLOT_REPLICATE_BARS:
                for m in metrics_ok:
                    plot_replicate_bars(
                        per_well_stats,
                        m,
                        os.path.join(plot_dir, f"{m}_replicateBars_withinSD.png"),
                    )
            if PLOT_CONDITION_MEAN:
                for m in metrics_ok:
                    plot_condition_mean_sem(
                        condition_summary,
                        m,
                        os.path.join(plot_dir, f"{m}_conditionMean_SEM.png"),
                    )
            print(f"[{exp_name}] Plots saved to: {plot_dir}")

    summary = pd.DataFrame({
        "Setting": [
            "Experiment",
            "InputDir",
            "PlatemapPath",
            "Composite_w_neurite",
            "Composite_w_cells",
            "USE_FILTER",
            "FILTER_MODE",
            "ENDPOINTS_MIN",
            "ENDPOINTS_MAX",
            "ENDPOINTS_PER_MPX_MIN",
            "ENDPOINTS_PER_MPX_MAX",
            "Total_Images",
            "Included_Count",
            "Excluded_Count",
            "ENABLE_PRECHECK",
            "N_SAMPLE_FOR_PRECHECK",
            "STOP_AFTER_PRECHECK",
            "SOMA_PERC",
            "SOMA_MIN_AREA",
            "SOMA_DILATE",
            "SOMA_THICK_RADIUS",
            "USE_SOMA_MAX_AREA_FILTER",
            "SOMA_MAX_AREA",
            "USE_SOMA_CIRCULARITY_FILTER",
            "SOMA_MIN_CIRCULARITY",
            "SOMA_MAX_CIRCULARITY",
            "VESSEL_WEIGHT",
            "VESSEL_MIN_RESP",
            "THRESHOLD_BIAS",
            "GLOBAL_MIN_T",
            "GLOBAL_MAX_T",
        ],
        "Value": [
            exp_name,
            input_dir,
            platemap_path,
            REPAIR_W_NEURITE,
            REPAIR_W_CELLS,
            USE_FILTER,
            FILTER_MODE,
            ENDPOINTS_MIN,
            ENDPOINTS_MAX,
            ENDPOINTS_PER_MPX_MIN,
            ENDPOINTS_PER_MPX_MAX,
            len(df),
            len(df_included),
            len(df_excluded),
            ENABLE_PRECHECK,
            N_SAMPLE_FOR_PRECHECK,
            STOP_AFTER_PRECHECK,
            SOMA_PERC,
            SOMA_MIN_AREA,
            SOMA_DILATE,
            SOMA_THICK_RADIUS,
            USE_SOMA_MAX_AREA_FILTER,
            SOMA_MAX_AREA,
            USE_SOMA_CIRCULARITY_FILTER,
            SOMA_MIN_CIRCULARITY,
            SOMA_MAX_CIRCULARITY,
            VESSEL_WEIGHT,
            VESSEL_MIN_RESP,
            THRESHOLD_BIAS,
            GLOBAL_MIN_T,
            GLOBAL_MAX_T,
        ],
    })

    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as xw:
        df.to_excel(xw, index=False, sheet_name="AllImages")
        df_included.to_excel(xw, index=False, sheet_name="Included")
        df_excluded.to_excel(xw, index=False, sheet_name="Excluded")
        summary.to_excel(xw, index=False, sheet_name="Summary")
        unmapped.to_excel(xw, index=False, sheet_name="UNMAPPED")
        parse_fail.to_excel(xw, index=False, sheet_name="PARSE_FAIL")

        if per_well_avg is not None:
            per_well_avg.to_excel(xw, index=False, sheet_name="PerWell_Avg")
        if per_well_stats is not None:
            per_well_stats.to_excel(xw, index=False, sheet_name="PerWell_Stats")
        if condition_summary is not None:
            condition_summary.to_excel(xw, index=False, sheet_name="Condition_Summary")

    return {
        "Experiment": exp_name,
        "InputDir": input_dir,
        "Images": len(image_paths),
        "Included": int(len(df_included)),
        "Excluded": int(len(df_excluded)),
        "OutputXLSX": out_xlsx,
        "OverlayDir": overlay_dir,
        "PlotDir": plot_dir if MAKE_PLOTS else "",
        "PlatemapPath": platemap_path,
    }


# ============================================================
# Main
# ============================================================
if __name__ == "__main__":
    os.makedirs(OUTPUT_ROOT, exist_ok=True)

    exp_folders = find_experiment_folders(ROOT_DIR)
    print(f"Found {len(exp_folders)} experiment folders under: {ROOT_DIR}")

    master_rows = []
    for folder in exp_folders:
        exp_name = os.path.basename(os.path.normpath(folder))
        platemap_path = pick_platemap_for_folder(folder)
        output_base = os.path.join(OUTPUT_ROOT, exp_name)

        try:
            res = run_one_experiment(folder, platemap_path, output_base)
            master_rows.append(res)
        except SystemExit as e:
            print(str(e))
            break
        except Exception as e:
            master_rows.append({"Experiment": exp_name, "InputDir": folder, "Error": repr(e)})
            print(f"FAILED {exp_name}: {e}")

    if len(master_rows) > 0:
        master_df = pd.DataFrame(master_rows)
        master_index_path = os.path.join(OUTPUT_ROOT, "_MASTER_INDEX.xlsx")
        master_df.to_excel(master_index_path, index=False)
        print(f"\nWrote master index: {master_index_path}")